# Lab 2: Basic Iceberg Operations - Solution

This notebook contains the complete solution for Lab 2. Use this to verify your implementation or if you get stuck.

## Step 1: Create Iceberg Tables with Different Configurations

Start Spark shell:
```bash
spark-shell \
  --packages org.apache.iceberg:iceberg-spark-runtime-3.5:1.5.0 \
  --conf spark.sql.catalog.iceberg=org.apache.iceberg.spark.SparkCatalog \
  --conf spark.sql.catalog.iceberg.type=rest \
  --conf spark.sql.catalog.iceberg.uri=http://localhost:8181/api/catalog \
  --conf spark.hadoop.fs.s3a.endpoint=http://localhost:9000 \
  --conf spark.hadoop.fs.s3a.access.key=minioadmin \
  --conf spark.hadoop.fs.s3a.secret.key=minioadmin \
  --conf spark.hadoop.fs.s3a.path.style.access=true
```

In [ ]:
// Table 1: Simple Unpartitioned Table
spark.sql("""
  CREATE TABLE iceberg.default.users (
    user_id INT,
    username STRING,
    email STRING,
    created_at TIMESTAMP
  ) USING iceberg
  TBLPROPERTIES (
    'format-version'='2',
    'write.format.default'='parquet',
    'write.metadata.compression-codec'='gzip'
  )
""")

In [ ]:
// Table 2: Partitioned Table
spark.sql("""
  CREATE TABLE iceberg.default.orders (
    order_id INT,
    user_id INT,
    order_date DATE,
    amount DECIMAL(10,2),
    status STRING
  ) USING iceberg
  PARTITIONED BY (order_date)
  TBLPROPERTIES (
    'format-version'='2',
    'write.format.default'='parquet'
  )
""")

In [ ]:
// Table 3: Multi-Level Partitioned Table
spark.sql("""
  CREATE TABLE iceberg.default.events (
    event_id STRING,
    user_id INT,
    event_timestamp TIMESTAMP,
    event_type STRING,
    region STRING,
    data STRING
  ) USING iceberg
  PARTITIONED BY (years(event_timestamp), region)
  TBLPROPERTIES (
    'format-version'='2',
    'write.format.default'='parquet'
  )
""")

In [ ]:
// Assertion: All three tables created successfully
spark.sql("SHOW TABLES IN iceberg.default").show()

val tables = spark.sql("SHOW TABLES IN iceberg.default").collect().map(_.getString(1))
assert(tables.contains("users"), "users table should exist")
assert(tables.contains("orders"), "orders table should exist")
assert(tables.contains("events"), "events table should exist")
println("✅ All tables created successfully!")

## Step 2: Insert Data into Tables

In [ ]:
// Insert data into users table
spark.sql("""
  INSERT INTO iceberg.default.users VALUES
    (1, 'alice', 'alice@example.com', TIMESTAMP '2024-01-01 00:00:00'),
    (2, 'bob', 'bob@example.com', TIMESTAMP '2024-01-02 00:00:00'),
    (3, 'charlie', 'charlie@example.com', TIMESTAMP '2024-01-03 00:00:00')
""")

In [ ]:
// Insert data into orders table
spark.sql("""
  INSERT INTO iceberg.default.orders VALUES
    (1, 1, DATE '2024-01-01', 100.50, 'completed'),
    (2, 1, DATE '2024-01-02', 250.75, 'completed'),
    (3, 2, DATE '2024-01-01', 75.25, 'pending'),
    (4, 2, DATE '2024-01-03', 150.00, 'shipped'),
    (5, 3, DATE '2024-01-02', 300.00, 'completed')
""")

In [ ]:
// Insert data into events table
spark.sql("""
  INSERT INTO iceberg.default.events VALUES
    ('evt001', 1, TIMESTAMP '2024-01-01 10:00:00', 'login', 'us-west', '{"device":"mobile"}'),
    ('evt002', 1, TIMESTAMP '2024-01-01 10:05:00', 'pageview', 'us-west', '{"page":"/home"}'),
    ('evt003', 2, TIMESTAMP '2024-01-01 11:00:00', 'login', 'us-east', '{"device":"desktop"}'),
    ('evt004', 2, TIMESTAMP '2024-01-02 09:00:00', 'purchase', 'us-east', '{"product":"item1"}')
""")

In [ ]:
// Verify data insertion
val userCount = spark.sql("SELECT COUNT(*) as user_count FROM iceberg.default.users").collect()(0).getLong(0)
val orderCount = spark.sql("SELECT COUNT(*) as order_count FROM iceberg.default.orders").collect()(0).getLong(0)
val eventCount = spark.sql("SELECT COUNT(*) as event_count FROM iceberg.default.events").collect()(0).getLong(0)

assert(userCount == 3, s"users should have 3 rows, but has $userCount")
assert(orderCount == 5, s"orders should have 5 rows, but has $orderCount")
assert(eventCount == 4, s"events should have 4 rows, but has $eventCount")
println("✅ Data inserted successfully!")

## Step 3: Query Data with Different Patterns

In [ ]:
// Simple query
spark.sql("SELECT * FROM iceberg.default.users WHERE username = 'alice'").show()

In [ ]:
// Join query
spark.sql("""
  SELECT u.username, o.order_id, o.amount, o.status
  FROM iceberg.default.users u
  JOIN iceberg.default.orders o ON u.user_id = o.user_id
  ORDER BY o.order_date
""").show()

In [ ]:
// Aggregation query
spark.sql("""
  SELECT 
    order_date,
    COUNT(*) as order_count,
    SUM(amount) as total_amount,
    AVG(amount) as avg_amount
  FROM iceberg.default.orders
  GROUP BY order_date
  ORDER BY order_date
""").show()

## Step 4: Schema Evolution

In [ ]:
// Add a new column to users table
spark.sql("""
  ALTER TABLE iceberg.default.users
  ADD COLUMN last_login TIMESTAMP
""")

In [ ]:
// Insert data with new column
spark.sql("""
  INSERT INTO iceberg.default.users VALUES
    (4, 'david', 'david@example.com', TIMESTAMP '2024-01-04 00:00:00', TIMESTAMP '2024-01-05 12:00:00')
""")

In [ ]:
// Verify new column exists and data is accessible
spark.sql("DESCRIBE iceberg.default.users").show()
spark.sql("SELECT * FROM iceberg.default.users WHERE user_id = 4").show()

## Step 5: Work with Snapshots

In [ ]:
// List snapshots for orders table
spark.sql("""
  SELECT 
    snapshot_id,
    committed_at,
    summary
  FROM iceberg.default.orders.snapshots
  ORDER BY committed_at DESC
""").show()

In [ ]:
// Get the latest snapshot ID for time travel
val snapshotId = spark.sql("""
  SELECT snapshot_id 
  FROM iceberg.default.orders.snapshots 
  ORDER BY committed_at DESC 
  LIMIT 1
""").collect()(0).getString(0)

// Time travel query
spark.sql(s"""
  SELECT * FROM iceberg.default.orders
  VERSION AS OF $snapshotId
""").show()

## Step 6: Update and Delete Operations

In [ ]:
// Update operation
spark.sql("""
  UPDATE iceberg.default.orders
  SET status = 'shipped'
  WHERE order_id = 3
""")

In [ ]:
// Verify update
spark.sql("SELECT * FROM iceberg.default.orders WHERE order_id = 3").show()

In [ ]:
// Delete operation
spark.sql("""
  DELETE FROM iceberg.default.users
  WHERE user_id = 4
""")

In [ ]:
// Verify deletion
spark.sql("SELECT * FROM iceberg.default.users WHERE user_id = 4").show()

val deletedCount = spark.sql("SELECT COUNT(*) FROM iceberg.default.users WHERE user_id = 4").collect()(0).getLong(0)
assert(deletedCount == 0, "Deleted row should not exist")
println("✅ Update and delete operations successful!")

## ✅ Lab Completion Checklist

- [x] Three tables created with different partitioning strategies
- [x] Data inserted into all tables successfully
- [x] Different query patterns executed successfully
- [x] Schema evolution performed without breaking existing data
- [x] Snapshot operations and time travel queries working
- [x] Update and delete operations successful

## 🎓 Key Concepts Learned

1. **Table Creation**: Different partitioning strategies (unpartitioned, single, multi-level)
2. **Data Operations**: Insert, update, delete with ACID guarantees
3. **Schema Evolution**: Add columns without breaking existing data
4. **Snapshots**: Time travel and rollback capabilities
5. **Query Patterns**: Joins, aggregations, and filtering with Iceberg tables